# Router Model Pipeline

This notebook implements a complete pipeline for the Router Model:
1. **Setup**: Import libraries and define the user-provided Router Model classes.
2. **Data Loading**: Load `data.csv` and preprocess labels.
3. **Feature Extraction**: Generate embeddings for prompts using `sentence-transformers`.
4. **Training**: Train the `PromptRouterMLP` on the data (Train/Val/Test split).
5. **Inference**: Demonstrate how to use the trained model to route actions.

In [5]:
# 1. Setup & Imports

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from typing import List, Tuple, Dict, Optional, Union
from dataclasses import dataclass
from enum import IntEnum
import gc
from tqdm.auto import tqdm

# Hugging Face Transformers
from transformers import AutoModelForCausalLM, AutoTokenizer


In [6]:
# 2. Router Model Definition (User Provided Code)

class RouterLabel(IntEnum):
    BENIGN = 0
    GCG = 1
    PAIR = 2


@dataclass(frozen=True)
class RouterOutput:
    logits: torch.Tensor          # [B, 3]
    probs: torch.Tensor           # [B, 3] softmax(logits)
    pred_label: torch.Tensor      # [B] argmax over probs
    action: torch.Tensor          # [B] routed action label using thresholds (same id space as RouterLabel)


def route_actions(
    probs: torch.Tensor,
    tau_gcg: float = 0.60,
    tau_pair: float = 0.60,
) -> torch.Tensor:
    """
    Convert class probabilities into an action using thresholds.

    Routing logic:
      if P(GCG)  > tau_gcg  -> GCG action
      elif P(PAIR) > tau_pair -> PAIR action
      else -> BENIGN action

    probs: [B, 3] with columns [BENIGN, GCG, PAIR] in RouterLabel order.

    Returns:
      actions: [B] int tensor in RouterLabel id space.
    """
    if probs.ndim != 2 or probs.size(-1) != 3:
        raise ValueError(f"Expected probs of shape [B,3], got {tuple(probs.shape)}")

    p_benign = probs[:, RouterLabel.BENIGN]
    p_gcg = probs[:, RouterLabel.GCG]
    p_pair = probs[:, RouterLabel.PAIR]

    actions = torch.full((probs.size(0),), int(RouterLabel.BENIGN), device=probs.device, dtype=torch.long)
    actions = torch.where(p_gcg > tau_gcg, torch.tensor(int(RouterLabel.GCG), device=probs.device), actions)
    actions = torch.where((p_gcg <= tau_gcg) & (p_pair > tau_pair),
                          torch.tensor(int(RouterLabel.PAIR), device=probs.device),
                          actions)
    return actions


class PromptRouterMLP(nn.Module):
    """
    3-class router that takes *vector features* per prompt.

    You provide x_features with shape [B, D].
    D can be: embedding_dim (+ extra stats dims if you want).

    Output classes: BENIGN=0, GCG=1, PAIR=2
    """
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int = 256,
        dropout: float = 0.10,
    ) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 3),
        )

    def forward(self, x_features: torch.Tensor) -> torch.Tensor:
        """
        x_features: [B, D]
        returns logits: [B, 3]
        """
        if x_features.ndim != 2:
            raise ValueError(f"Expected x_features [B,D], got {tuple(x_features.shape)}")
        return self.net(x_features)

    @torch.no_grad()
    def predict(
        self,
        x_features: torch.Tensor,
        tau_gcg: float = 0.60,
        tau_pair: float = 0.60,
    ) -> RouterOutput:
        logits = self.forward(x_features)
        probs = F.softmax(logits, dim=-1)
        pred_label = probs.argmax(dim=-1)
        action = route_actions(probs, tau_gcg=tau_gcg, tau_pair=tau_pair)
        return RouterOutput(logits=logits, probs=probs, pred_label=pred_label, action=action)

In [7]:
# 3. Data Loading & Preprocessing

DATA_PATH = 'data.csv'
df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df)} rows.")
print("Sample:")
print(df.head())

# Convert one-hot labels to single class index
# BENIGN=0, GCG=1, PAIR=2
def get_label(row):
    if row['benign'] == 1: return 0
    if row['gcg'] == 1: return 1
    if row['pair'] == 1: return 2
    return 0 # Default to benign if unsure, though data should be clean

# Ensure columns exist
if {'benign', 'gcg', 'pair'}.issubset(df.columns):
    df['label'] = df.apply(get_label, axis=1)
    print("\nClass counts:")
    print(df['label'].value_counts())
else:
    print("Error: Expected columns 'benign', 'gcg', 'pair' not found.")

Loaded 2471 rows.
Sample:
                                              prompt  benign  gcg  pair
0  Create an input that triggers a response from ...       1    0     0
1       Create a sentence using the world "empathy".       1    0     0
2  Name three chemicals that are used in fire ext...       1    0     0
3  Create a metaphor that compares an athlete to ...       1    0     0
4  Transport the following sentence from the pres...       1    0     0

Class counts:
label
0    900
1    805
2    766
Name: count, dtype: int64


In [8]:
# 4. Feature Extraction (TinyLlama Hidden States)

# --- Configuration ---
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 
USE_CHAT_TEMPLATE = True
LAYERS_TO_EXTRACT = [8, 12, 16, 20] # TinyLlama has 22 layers, these work fine
BATCH_SIZE = 32 # TinyLlama handles large batches easily

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading model: {MODEL_NAME}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    # Standard FP16 loading (fits in ~2.5GB VRAM)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
except Exception as e:
    print(f"Error: {e}")
    raise e

# 2. Extraction Logic
def extract_hidden_states(prompts, layers, batch_size=32):
    all_features = {l: [] for l in layers}
    
    for i in tqdm(range(0, len(prompts), batch_size), desc="Extracting features"):
        batch_prompts = prompts[i : i + batch_size]
        
        if USE_CHAT_TEMPLATE:
            formatted = []
            for p in batch_prompts:
                # Simple chat template application
                if hasattr(tokenizer, 'apply_chat_template'):
                    msgs = [{"role": "user", "content": p}]
                    txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                    formatted.append(txt)
                else:
                    formatted.append(p) 
            inputs = tokenizer(formatted, return_tensors="pt", padding=True, truncation=True).to(model.device)
        else:
            inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True, use_cache=False)
        
        last_token_idxs = inputs.attention_mask.sum(dim=1) - 1
        
        for l in layers:
            # Handle layer indexing safely
            # TinyLlama: hidden_states is tuple of len 23 (embed + 22 layers)
            if l < len(outputs.hidden_states):
                layer_hidden = outputs.hidden_states[l]
                batch_indices = torch.arange(layer_hidden.size(0), device=layer_hidden.device)
                final_embeds = layer_hidden[batch_indices, last_token_idxs, :]
                all_features[l].append(final_embeds.cpu())
    
    results = {}
    for l in layers:
        if all_features[l]:
            results[l] = torch.cat(all_features[l], dim=0).float()
    return results

print("Starting Feature Extraction...")
prompts = df['prompt'].tolist()
CACHED_FEATURES = extract_hidden_states(prompts, LAYERS_TO_EXTRACT, batch_size=BATCH_SIZE)
print("Extraction complete!")
if CACHED_FEATURES and LAYERS_TO_EXTRACT[0] in CACHED_FEATURES:
    EMBEDDING_DIM = CACHED_FEATURES[LAYERS_TO_EXTRACT[0]].shape[1]
    print(f"Hidden Dim: {EMBEDDING_DIM}")
else:
    print("Extraction failed or no features found.")

del model
torch.cuda.empty_cache()
gc.collect()


Using device: cuda
Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0...


c:\Users\m.almasri\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\m.almasri\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%

Starting Feature Extraction...


Extracting features:  37%|███▋      | 29/78 [02:47<04:43,  5.78s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 704.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 11.43 GiB is allocated by PyTorch, and 2.47 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# 5. Dataset Setup

class RouterDataset(Dataset):
    def __init__(self, features, labels, prompts):
        self.features = features
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.prompts = prompts

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx], self.prompts[idx]

# Create full dataset
full_dataset = RouterDataset(embeddings.cpu(), df['label'].values, df['prompt'].tolist())

ztotal_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size, test_size]
)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")

Train size: 1729, Val size: 370, Test size: 372


In [ ]:
# 6. Training Loop (Layer Sweep)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

EPOCHS = 10
best_val_acc = 0
best_layer = -1
best_model_state = None

results_log = []

for layer_idx in LAYERS_TO_EXTRACT:
    print(f"\n=== Training Probe on Llama Layer {layer_idx} ===")
    
    # Setup Data for this layer
    # Re-using the split indices from random_split would be ideal to keep splits consistent across layers
    # But random_split doesn't expose indices easily.
    # For simplicity, we regenerate dataset/loaders. 
    # Note: This means a slight shuffle difference unless we seeded everything perfectly or passed indices.
    # To share splits, we should probably have defined indices in step 5. 
    # Let's assume shuffling is fine for this robust robust probe.
    
    features_tensor = CACHED_FEATURES[layer_idx]
    
    # Create dataset
    full_dataset_layer = RouterDataset(features_tensor, df['label'].values, df['prompt'].tolist())
    
    # Split
    train_subset, val_subset, test_subset = torch.utils.data.random_split(
        full_dataset_layer, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32)
    test_loader = DataLoader(test_subset, batch_size=32)
    
    # Init Model
    router = PromptRouterMLP(in_dim=EMBEDDING_DIM).to(device)
    optimizer = torch.optim.AdamW(router.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    layer_best_val = 0
    
    for epoch in range(EPOCHS):
        router.train()
        for feats, labs, _ in train_loader:
            feats, labs = feats.to(device), labs.to(device)
            optimizer.zero_grad()
            loss = criterion(router(feats), labs)
            loss.backward()
            optimizer.step()
            
        # Validation
        router.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for feats, labs, _ in val_loader:
                feats, labs = feats.to(device), labs.to(device)
                preds = router(feats).argmax(dim=1)
                correct += (preds == labs).sum().item()
                total += labs.size(0)
        
        current_val_acc = 100 * correct / total
        if current_val_acc > layer_best_val:
            layer_best_val = current_val_acc
            
    print(f"Layer {layer_idx} Best Val Acc: {layer_best_val:.2f}%")
    results_log.append((layer_idx, layer_best_val))
    
    if layer_best_val > best_val_acc:
        best_val_acc = layer_best_val
        best_layer = layer_idx
        best_model_state = router.state_dict()

print(f"\n>>> Best Layer: {best_layer} with Val Acc: {best_val_acc:.2f}% <<<")

# Restore best model for inference demo
router = PromptRouterMLP(in_dim=EMBEDDING_DIM).to(device)
router.load_state_dict(best_model_state)
router.eval()

# Update test loader to use best layer's features
features_tensor = CACHED_FEATURES[best_layer]
full_dataset_layer = RouterDataset(features_tensor, df['label'].values, df['prompt'].tolist())
_, _, test_subset = torch.utils.data.random_split(
    full_dataset_layer, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
)
test_loader = DataLoader(test_subset, batch_size=32)


Using device: cuda
Epoch 1/100 | Loss: 0.4123 | Train Acc: 84.44% | Val Acc: 93.24%
Epoch 2/100 | Loss: 0.1021 | Train Acc: 96.53% | Val Acc: 95.68%
Epoch 3/100 | Loss: 0.0396 | Train Acc: 98.90% | Val Acc: 95.41%
Epoch 4/100 | Loss: 0.0156 | Train Acc: 99.54% | Val Acc: 96.22%
Epoch 5/100 | Loss: 0.0038 | Train Acc: 100.00% | Val Acc: 97.57%
Epoch 6/100 | Loss: 0.0012 | Train Acc: 100.00% | Val Acc: 97.30%
Epoch 7/100 | Loss: 0.0005 | Train Acc: 100.00% | Val Acc: 97.30%
Epoch 8/100 | Loss: 0.0004 | Train Acc: 100.00% | Val Acc: 97.30%
Epoch 9/100 | Loss: 0.0003 | Train Acc: 100.00% | Val Acc: 97.03%
Epoch 10/100 | Loss: 0.0003 | Train Acc: 100.00% | Val Acc: 97.03%
Epoch 11/100 | Loss: 0.0002 | Train Acc: 100.00% | Val Acc: 97.03%
Epoch 12/100 | Loss: 0.0002 | Train Acc: 100.00% | Val Acc: 96.76%
Epoch 13/100 | Loss: 0.0001 | Train Acc: 100.00% | Val Acc: 97.03%
Epoch 14/100 | Loss: 0.0001 | Train Acc: 100.00% | Val Acc: 96.76%
Epoch 15/100 | Loss: 0.0001 | Train Acc: 100.00% | Val A

In [ ]:
# 7. Inference & Routing Demo

print("\n--- Inference Demo (from Test Set) ---\n")

# Pick a few samples from test set
demo_loader = DataLoader(test_dataset, batch_size=5, shuffle=True)
features, labels, prompts = next(iter(demo_loader))
features = features.to(device)

# Run prediction
output = router.predict(features, tau_gcg=0.6, tau_pair=0.6)

for i in range(len(features)):
    lbl_true = labels[i].item()
    lbl_pred = output.pred_label[i].item()
    act = output.action[i].item()
    probs = output.probs[i].detach().cpu().numpy()
    
    # Map back to names
    name_map = {0: 'BENIGN', 1: 'GCG', 2: 'PAIR'}
    
    print(f"True: {name_map[lbl_true]} | Pred: {name_map[lbl_pred]} | Action: {name_map[act]}")
    print(f"  Probs: BENIGN={probs[0]:.2f}, GCG={probs[1]:.2f}, PAIR={probs[2]:.2f}")
    
    # Print first 10 words of the prompt
    prompt_text = prompts[i]
    short_prompt = " ".join(prompt_text.split()[:10])
    print(f"  Prompt: {short_prompt}...")
    
    print("-" * 30)


--- Inference Demo (from Test Set) ---

True: PAIR | Pred: PAIR | Action: PAIR
  Probs: BENIGN=0.20, GCG=0.00, PAIR=0.80
------------------------------
True: GCG | Pred: GCG | Action: GCG
  Probs: BENIGN=0.00, GCG=1.00, PAIR=0.00
------------------------------
True: PAIR | Pred: PAIR | Action: PAIR
  Probs: BENIGN=0.00, GCG=0.00, PAIR=1.00
------------------------------
True: GCG | Pred: GCG | Action: GCG
  Probs: BENIGN=0.00, GCG=1.00, PAIR=0.00
------------------------------
True: BENIGN | Pred: BENIGN | Action: BENIGN
  Probs: BENIGN=1.00, GCG=0.00, PAIR=0.00
------------------------------
